In [ ]:
%pip install pandas quinex click quinex-utils tdqm

In [ ]:
!python3 -m spacy download en_core_web_md

In [ ]:
%pip install git+https://github.com/FZJ-IEK3-VSA/llm-chunker.git

In [ ]:
from quinex import Quinex
import pandas as pd
from tqdm import tqdm

DESCRIPTION_CSV = "cleaned_labeled_descriptions.csv"

description_df = pd.read_csv(DESCRIPTION_CSV)

quinex = Quinex()

In [ ]:
import pandas as pd

# in case a key is missing in a nested dictionary, return None instead of raising an error
def safe_get(d, *keys):
    for key in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(key)
    return d

import pandas as pd


def parse_quinex_results(results):
    # if Quinex returns no results, return an empty DataFrame
    if not results:
        return pd.DataFrame()

    rows = []

    for record in results:

        claim = record.get("claim") or {}

        entity = (claim.get("entity") or {}).get("text")
        property_name = (claim.get("property") or {}).get("text")

        quantity = claim.get("quantity") or {}

        quantity_text = quantity.get("text")

        normalized = quantity.get("normalized") or {}

        quantity_type = normalized.get("type")

        individual_quantities = (
            (normalized.get("individual_quantities") or {})
            .get("normalized", [])
        )

        values = []
        unit = None

        for q in individual_quantities:

            numeric_value = (
                ((q.get("value") or {})
                 .get("normalized") or {})
                .get("numeric_value")
            )

            if numeric_value is not None:
                values.append(numeric_value)

            if unit is None:
                unit_info = (q.get("unit") or {})
                normalized_units = unit_info.get("normalized") or []

                if normalized_units:
                    unit = normalized_units[0].get("text")
                else:
                    # fall back to raw unit text
                    text_info = unit_info.get("text") or {}

                    unit = (
                        text_info.get("suffixed")
                        or text_info.get("ellipsed")
                    )

        rows.append({
            "entity": entity,
            "property": property_name,
            "value_min": min(values) if values else None,
            "value_max": max(values) if values else None,
            "unit": unit,
            "quantity_text": quantity_text,
            "quantity_type": quantity_type,
        })

    return pd.DataFrame(rows)

In [ ]:
#Sanity check
text = description_df[description_df["detected_language"] == "ENGLISH"]["cleaned_description"].iloc[110]

qclaims = quinex(text)

print(text)
print(parse_quinex_results(qclaims))

In [ ]:
import os
import pickle
import pandas as pd
from tqdm import tqdm

CHECKPOINT_DATA = "quinex_results.pkl"
CHECKPOINT_STATE = "quinex_state.pkl"
SAVE_EVERY = 10

In [ ]:
if os.path.exists(CHECKPOINT_DATA) and os.path.exists(CHECKPOINT_STATE):

    result_df = pd.read_pickle(CHECKPOINT_DATA)

    with open(CHECKPOINT_STATE, "rb") as f:
        start_idx = pickle.load(f)

    print(f"Resuming from row {start_idx:,}")

else:
    result_df = pd.DataFrame()
    start_idx = 0

english_descriptions = (
    description_df[description_df["detected_language"] == "ENGLISH"]
    .reset_index(drop=True)
)

for idx in tqdm(
    range(start_idx, len(english_descriptions)),
    total=len(english_descriptions),
    initial=start_idx,
):

    row = english_descriptions.iloc[idx]

    try:
        qclaims = quinex(row.cleaned_description)

        parsed_df = parse_quinex_results(qclaims)

        if not parsed_df.empty:
            parsed_df["ipni_id"] = row.ipni_id
            parsed_df["provider"] = row.provider

            result_df = pd.concat(
                [result_df, parsed_df],
                ignore_index=True,
            )

    except Exception as e:
        print(f"Error at row {idx}: {e}")

    if (idx + 1) % SAVE_EVERY == 0:

        result_df.to_pickle(CHECKPOINT_DATA)

        with open(CHECKPOINT_STATE, "wb") as f:
            pickle.dump(idx + 1, f)

result_df.to_pickle(CHECKPOINT_DATA)

print(f"Final checkpoint data saved to {CHECKPOINT_DATA}")

with open(CHECKPOINT_STATE, "wb") as f:
    pickle.dump(len(english_descriptions), f)

print(f"Final checkpoint state saved to {CHECKPOINT_STATE}")

: 

In [ ]:
result_df = pd.read_pickle("quinex_results.pkl")

result_df.head()

result_df.to_csv("extracted_quantities.csv", index=False)